# Đoạn code thực hiện yêu cầu bài tập chương 2  

In [1]:
import sqlite3
import pandas as pd
import datetime

# Khởi tạo database trong bộ nhớ
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Tạo bảng student và course
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);
""")

# Thêm dữ liệu
students = [
    (1,"Nguyen Minh Hoang","May Tinh",12,6.7),
    (2,"Tran Thi Lan","Kinh Te",34,9.2),
    (3,"Pham Van Nam","Toan Tin",20,7.9),
    (4,"Le Thanh Huyen","Toan Tin",20,7.2),
    (5,"Vu Quoc Anh","May Tinh",24,8.0),
    (6,"Dang Thuy Linh","May Tinh",24,5.5),
    (7,"Bui Tien Dung","Kinh Te",34,9.2),
    (8,"Ho Ngoc Mai","Toan Tin",20,8.8),
    (9,"Duong Huu Phuc","Kinh Te",20,7.2),
    (10,"Cao Thi Hanh","May Tinh",20,7.0),
]
cursor.executemany("INSERT INTO student VALUES (?,?,?,?,?)", students)

courses = [(12,"Giai tich"),(34,"Thong ke"),(26,"Tin hoc")]
cursor.executemany("INSERT INTO course VALUES (?,?)", courses)
conn.commit()

# 1. JOINs
print("Tích Descartes:")
df_cartesian = pd.read_sql_query("SELECT * FROM student, course;", conn)
print(df_cartesian)

print("\nINNER JOIN:")
df_inner = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
INNER JOIN course c ON s.course_id = c.id;
""", conn)
print(df_inner)

print("\nLEFT JOIN:")
df_left = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id;
""", conn)
print(df_left)

print("\nRIGHT JOIN (mô phỏng):")
df_right = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM course c
LEFT JOIN student s ON s.course_id = c.id;
""", conn)
print(df_right)

print("\nFULL OUTER JOIN (mô phỏng):")
df_full = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM course c
LEFT JOIN student s ON s.course_id = c.id;
""", conn)
print(df_full)

# 2. Loại bỏ bản ghi không hợp lệ
cursor.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course);")
conn.commit()

# 2a. Thống kê theo lớp
df_class = pd.read_sql_query("""
SELECT class, COUNT(*) AS total_students, AVG(score) AS avg_score
FROM student
GROUP BY class;
""", conn)
print("\nTheo lớp:\n", df_class)

# 2b. Thống kê theo môn học
df_course = pd.read_sql_query("""
SELECT c.course_name, COUNT(*) AS total_students, AVG(s.score) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name;
""", conn)
print("\nTheo môn học:\n", df_course)

# 2c. Phân loại thi đua
df_rank = pd.read_sql_query("""
SELECT c.course_name,
       CASE
         WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
         WHEN AVG(s.score) BETWEEN 6.0 AND 8.9 THEN 'Tốt'
         ELSE 'Kém'
       END AS rank
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name;
""", conn)
print("\nPhân loại thi đua:\n", df_rank)

# 3. Xếp hạng sinh viên
df_rank_all = pd.read_sql_query("""
SELECT student_id, name, score,
       RANK() OVER (ORDER BY score DESC) AS rank
FROM student;
""", conn)
print("\nXếp hạng toàn bộ:\n", df_rank_all)

print("\nTop 3 cao nhất:\n", df_rank_all.nsmallest(3, "rank"))
print("\nTop 3 thấp nhất:\n", df_rank_all.nlargest(3, "rank"))

# Theo lớp
df_rank_class = pd.read_sql_query("""
SELECT student_id, name, class, score,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student;
""", conn)
print("\nXếp hạng theo lớp:\n", df_rank_class)

# Theo môn học
df_rank_course = pd.read_sql_query("""
SELECT student_id, name, course_id, score,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank
FROM student;
""", conn)
print("\nXếp hạng theo môn học:\n", df_rank_course)

# 4. Thêm graduation_date
cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME;")
today = datetime.date.today()
for row in df_rank_all.itertuples():
    grad_date = today + datetime.timedelta(days=row.rank)
    cursor.execute("UPDATE student SET graduation_date=? WHERE student_id=?",
                   (grad_date, row.student_id))
conn.commit()

df_final = pd.read_sql_query("SELECT * FROM student;", conn)
print("\nBảng student cuối cùng:\n", df_final)


Tích Descartes:
    student_id               name     class  course_id  score  id course_name
0            1  Nguyen Minh Hoang  May Tinh         12    6.7  12   Giai tich
1            1  Nguyen Minh Hoang  May Tinh         12    6.7  26     Tin hoc
2            1  Nguyen Minh Hoang  May Tinh         12    6.7  34    Thong ke
3            2       Tran Thi Lan   Kinh Te         34    9.2  12   Giai tich
4            2       Tran Thi Lan   Kinh Te         34    9.2  26     Tin hoc
5            2       Tran Thi Lan   Kinh Te         34    9.2  34    Thong ke
6            3       Pham Van Nam  Toan Tin         20    7.9  12   Giai tich
7            3       Pham Van Nam  Toan Tin         20    7.9  26     Tin hoc
8            3       Pham Van Nam  Toan Tin         20    7.9  34    Thong ke
9            4     Le Thanh Huyen  Toan Tin         20    7.2  12   Giai tich
10           4     Le Thanh Huyen  Toan Tin         20    7.2  26     Tin hoc
11           4     Le Thanh Huyen  Toan Tin     

C:\Users\Admin\AppData\Local\Temp\ipykernel_12340\1411624487.py:154: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("UPDATE student SET graduation_date=? WHERE student_id=?",
